# Feature Selection using Chi Square test

In [1]:
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

In [8]:
dataset = fetch_openml(name='car')
X = dataset.data
Y = dataset.target

c:\Users\shash\anaconda3\envs\convo_env\Lib\site-packages\sklearn\datasets\_openml.py:328: UserWarning: Multiple active versions of the dataset matching the name car exist. Versions may be fundamentally different, returning version 2. Available versions:
- version 2, status: active
  url: https://www.openml.org/search?type=data&id=991
- version 3, status: active
  url: https://www.openml.org/search?type=data&id=40975

  warn(warning_msg)


In [10]:
X['target'] = Y

In [11]:
X

,buying,maint,doors,persons,lug_boot,safety,target
0,vhigh,vhigh,2,2,small,low,P
1,vhigh,vhigh,2,2,small,med,P
2,vhigh,vhigh,2,2,small,high,P
3,vhigh,vhigh,2,2,med,low,P
4,vhigh,vhigh,2,2,med,med,P
...,...,...,...,...,...,...,...
1723,low,low,5more,more,med,med,N
1724,low,low,5more,more,med,high,N
1725,low,low,5more,more,big,low,P
1726,low,low,5more,more,big,med,N


In [12]:
X['doors'].unique()

['2', '3', '4', '5more']
Categories (4, str): ['2', '3', '4', '5more']

In [13]:
Y

0       P
1       P
2       P
3       P
4       P
       ..
1723    N
1724    N
1725    P
1726    N
1727    N
Name: binaryClass, Length: 1728, dtype: category
Categories (2, str): ['N', 'P']

In [24]:
def chi2(df,attr,target):
    attr_values = df[attr].unique()
    target_values = df[target].unique()
    chi2_score = 0

    for attr_value in attr_values:
        for target_value in target_values:
            observed_freq = len(df[(df[attr]==attr_value)&(df[target]==target_value)]) # len as we want the frequency
            attr_prob =  len(df[df[attr]==attr_value])/len(df)
            target_prob = len(df[df[target]==target_value])/len(df)
            expected_freq = len(df) * (attr_prob * target_prob)

            chi2_score += ((observed_freq-expected_freq)**2)/expected_freq

    return chi2_score #12m, 8 mcq ( each of 1/2 marks), 8 questions of 1m each code based question ( jo notebook sir upload kar rhe hain for self work will form the mcqs)


In [16]:
X['buying'].unique()

['vhigh', 'high', 'med', 'low']
Categories (4, str): ['high', 'low', 'med', 'vhigh']

In [17]:
Y.unique()

['P', 'N']
Categories (2, str): ['N', 'P']

In [25]:
chi2(X,'buying','target')

76.5218545582182

In [30]:
def SelectKBest_scratch(df,k):
    attributes = df.columns[:-1] # all except -1
    target = df.columns[-1]
    chi2_score = {}
    for attr in attributes:
        chi2_score[attr] = chi2(df,attr,target)

    sorted_chi2_score = sorted(chi2_score.items(),key= lambda x: x[1], reverse=True) # default sorting order is asc
    selected_feature = [f for f,_ in sorted_chi2_score[:k]]
    return selected_feature,sorted_chi2_score

In [32]:
SelectKBest_scratch(X,3)

(['safety', 'persons', 'buying'],
 [('safety', 396.34438878075235),
  ('persons', 370.2912281821373),
  ('buying', 76.5218545582182),
  ('maint', 64.17072657072656),
  ('lug_boot', 29.400095727368452),
  ('doors', 8.590650627014263)])

In [ ]:
# in built methods :> scipy and sklearn , scipy is the same as the scratch one above but sklearn uses some sort of normalization
from scipy.stats import chi2_contingency
table = pd.crosstab(X.iloc[:,0],X.iloc[:,-1]) # crosstab() function is used to make the contigency table

In [36]:
table

target,N,P
buying,,
high,108,324
low,174,258
med,164,268
vhigh,72,360


In [37]:
chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(76.5218545582182), pvalue=np.float64(1.7095876176351842e-16), dof=3, expected_freq=array([[129.5, 302.5],
       [129.5, 302.5],
       [129.5, 302.5],
       [129.5, 302.5]]))

In [38]:
def SelectKBest_inbuilt(df,k):
    attributes = df.columns[:-1]
    target = df.columns[-1]
    chi2_score = {}
    for attr in attributes:
        table = pd.crosstab(df[attr],df[target])
        chi2,_,_,_ = chi2_contingency(table) # We only want the first value from what this function returns so we used "_"
        chi2_score[attr] = chi2
    sorted_chi2_score = sorted(chi2_score.items(), key= lambda x:x[1],reverse=True)
    selected_features = [f for f,_ in sorted_chi2_score[:k]]
    return selected_features,sorted_chi2_score


In [39]:
SelectKBest_inbuilt(X,3)

(['safety', 'persons', 'buying'],
 [('safety', np.float64(396.34438878075247)),
  ('persons', np.float64(370.2912281821373)),
  ('buying', np.float64(76.5218545582182)),
  ('maint', np.float64(64.17072657072657)),
  ('lug_boot', np.float64(29.400095727368456)),
  ('doors', np.float64(8.590650627014263))])

sklearn approach ( dont prefer), accepts only numerical values

In [43]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for attribute in X.columns:
    X[attribute] = le.fit_transform(X[attribute])

In [45]:
from sklearn.feature_selection import chi2,SelectKBest
test = SelectKBest(chi2,k=3)
test.fit(X.iloc[:,:-1],X.iloc[:,-1])
test.scores_

array([  6.39795782,   8.75055362,   5.55984556, 177.86727081,
        18.53771977,  17.64446855])

In [48]:
X.columns[:-1][test.get_support()] # from the input features we are taking the best , the answer will be different :> here it is wrong, so prefer to use scipy

Index(['persons', 'lug_boot', 'safety'], dtype='str')

# Feature Selection using Information Gain

In [ ]:
def find_entropy(df):
    target = df.columns[-1]
    target_values = df[target].unique()
    entropy = 0

    for target_value in target_values:
        prob = len(df[df[target]==target_value])/len(df)
        entropy += -prob * np.log(prob) # the inbuilt method takes natural log, so we have not taken log2() here to match
    return entropy

In [50]:
find_entropy(X)

np.float64(0.610668040691973)

In [55]:
def avg_info_entropy(df,attr):
    attr_values = df[attr].unique()
    target = df.columns[-1]
    target_values = df[target].unique()
    avg_info_entropy = 0

    for attr_value in attr_values:
        entropy_subsample = 0
        for target_value in target_values:
            num = len(df[(df[attr]==attr_value)&(df[target]==target_value)])
            den = len(df[attr] == attr_value)
            prob = num/den 
            entropy_subsample += -prob*np.log(prob+1e-7) #1e-7 is added to make sure the prob is not zero in case that particular instance is not present

        weight = len(df[df[attr]==attr_value])/len(df)
        avg_info_entropy += weight * entropy_subsample

    return avg_info_entropy


In [ ]:
find_entropy(X) - avg_info_entropy(X,'buying') #my answer is wrong, fix the code

np.float64(0.11716315222981044)

In [56]:
def SelectKBest_scratch(df,k):
    attributes = df.columns[:-1] # all except -1
    target = df.columns[-1]
    IG_score = {}
    for attr in attributes:
        IG_score[attr] = find_entropy(df) - avg_info_entropy(df,attr)

    sorted_IG_score = sorted(IG_score.items(),key= lambda x: x[1], reverse=True) # default sorting order is asc
    selected_feature = [f for f,_ in sorted_IG_score[:k]]
    return selected_feature,sorted_IG_score

In [57]:
SelectKBest_scratch(X,3)

(['buying', 'maint', 'doors'],
 [('buying', np.float64(0.11716335222970575)),
  ('maint', np.float64(0.11630144852354107)),
  ('doors', np.float64(0.11206374312734779)),
  ('safety', np.float64(0.09373259791466848)),
  ('persons', np.float64(0.09159374736149428)),
  ('lug_boot', np.float64(0.04381679826150031))])

# Inbuilt method

In [61]:
from sklearn.metrics import mutual_info_score
def SelectKBest_inbuilt(df,k):
    attributes = df.columns[:-1]
    IG_score_inbuilt = {}

    for attr in attributes:
        IG_score_inbuilt[attr] = mutual_info_score(df[attr],df.iloc[:,-1])

    sorted_IG_score_inbuilt = sorted(IG_score_inbuilt.items(),key= lambda x: x[1], reverse=True) 
    selected_feature = [f for f,_ in sorted_IG_score_inbuilt[:k]]
    return selected_feature,sorted_IG_score_inbuilt
    

In [62]:
SelectKBest_inbuilt(X,3)

(['safety', 'persons', 'buying'],
 [('safety', 0.15847350102830657),
  ('persons', 0.15205694936878172),
  ('buying', 0.022942847963213067),
  ('maint', 0.019495233138550103),
  ('lug_boot', 0.008726002068885186),
  ('doors', 0.0025444115537469336)])

In [63]:
from sklearn.feature_selection import SelectKBest,mutual_info_classif
test = SelectKBest(mutual_info_classif,k=3)
test.fit(X.iloc[:,:-1],X.iloc[:,-1])
X.columns[:-1][test.get_support()]

Index(['buying', 'persons', 'safety'], dtype='str')

In [64]:
test.scores_

array([0.02137731, 0.01570232, 0.        , 0.16537039, 0.01468   ,
       0.15083055])